# Supply Chain Control Tower – Risk Monitoring Layer

## Objective

This notebook develops the operational risk monitoring layer for the Supply Chain Control Tower.

The objective is to transform transactional demand data into actionable supply chain risk indicators that support capacity planning, bottleneck identification, inventory monitoring, and operational decision-making.

## Scope

Current Risk Modules:

- Capacity Risk

Future Risk Modules:

- Demand Risk
- Inventory Risk
- Service Risk
- Profitability Risk

## Business Questions

This notebook aims to answer the following questions:

1. Which manufacturing plants are approaching capacity constraints?

2. Where are potential bottlenecks occurring?

3. What products are driving capacity utilization increases?

4. Which operational risks require management attention?

## Data Sources

Master Data:
- dim_product
- dim_plant
- product_plant_mapping
- capacity_table

Processed Data:
- weekly_plant_demand

## Deliverables

- Capacity Utilization KPI
- Available Capacity KPI
- Capacity Alert Logic
- Bottleneck Monitoring Dataset
- Root Cause Analysis Dataset

The outputs generated in this notebook will serve as the analytical foundation for the Supply Chain Control Tower Dashboard.

### ==========================================
### Load Data
### ==========================================

In [34]:
import pandas as pd

In [46]:
weekly_plant_demand = pd.read_csv(
    "../data/processed/weekly_plant_demand.csv"
)

capacity_table = pd.read_csv(
    "../data/master/capacity_table.csv"
)

df = pd.read_csv(
    "../data/raw/DataCoSupplyChainDataset.csv",
    encoding="latin1"
)

product_plant_mapping = pd.read_csv(
    "../data/master/product_plant_mapping.csv"
)

In [36]:
capacity_utilization = (
    weekly_plant_demand.merge(
        capacity_table[
            ["Plant_ID", "Weekly_Capacity"]
        ],
        on="Plant_ID",
        how="left"
    )
)

capacity_utilization.head()

,Plant_ID,Order_Date,Order Item Quantity,Weekly_Capacity
0,P1,2015-01-04,567,1165
1,P1,2015-01-11,969,1165
2,P1,2015-01-18,988,1165
3,P1,2015-01-25,1009,1165
4,P1,2015-02-01,963,1165


In [37]:
# Total demand assigned to a plant in a given week

capacity_utilization = capacity_utilization.rename(
    columns={
        "Order Item Quantity": "Weekly_Demand"
    }
)

In [38]:
# Percentage of plant capacity consumed by weekly demand

capacity_utilization["Utilization"] = (
    capacity_utilization["Weekly_Demand"]
    /
    capacity_utilization["Weekly_Capacity"]
)

In [39]:
# Remaining production capacity available after fulfilling weekly demand

capacity_utilization["Available_Capacity"] = (
    capacity_utilization["Weekly_Capacity"]
    -
    capacity_utilization["Weekly_Demand"]
)

In [40]:
# Capacity risk level based on utilization threshold
def capacity_status(util):

    if util >= 0.95:
        return "Critical"

    elif util >= 0.85:
        return "Warning"

    else:
        return "Normal"

capacity_utilization["Capacity_Status"] = (
    capacity_utilization["Utilization"]
    .apply(capacity_status)
)

In [41]:
capacity_utilization.head()
capacity_utilization

,Plant_ID,Order_Date,Weekly_Demand,Weekly_Capacity,Utilization,Available_Capacity,Capacity_Status
0,P1,2015-01-04,567,1165,0.486695,598,Normal
1,P1,2015-01-11,969,1165,0.831760,196,Normal
2,P1,2015-01-18,988,1165,0.848069,177,Normal
3,P1,2015-01-25,1009,1165,0.866094,156,Warning
4,P1,2015-02-01,963,1165,0.826609,202,Normal
...,...,...,...,...,...,...,...
481,P3,2018-01-07,352,913,0.385542,561,Normal
482,P3,2018-01-14,269,913,0.294633,644,Normal
483,P3,2018-01-21,479,913,0.524644,434,Normal
484,P3,2018-01-28,304,913,0.332968,609,Normal


In [42]:
capacity_utilization

,Plant_ID,Order_Date,Weekly_Demand,Weekly_Capacity,Utilization,Available_Capacity,Capacity_Status
0,P1,2015-01-04,567,1165,0.486695,598,Normal
1,P1,2015-01-11,969,1165,0.831760,196,Normal
2,P1,2015-01-18,988,1165,0.848069,177,Normal
3,P1,2015-01-25,1009,1165,0.866094,156,Warning
4,P1,2015-02-01,963,1165,0.826609,202,Normal
...,...,...,...,...,...,...,...
481,P3,2018-01-07,352,913,0.385542,561,Normal
482,P3,2018-01-14,269,913,0.294633,644,Normal
483,P3,2018-01-21,479,913,0.524644,434,Normal
484,P3,2018-01-28,304,913,0.332968,609,Normal


In [43]:
capacity_utilization.to_csv(
    "../data/processed/capacity_utilization_weekly.csv",
    index=False
)

### ==========================================
### Product Root Cause Analysis Dataset
### ==========================================

In [47]:
plant_product_weekly_demand = (
    df.merge(
        product_plant_mapping[
            [
                "Product Card Id",
                "Plant_ID"
            ]
        ],
        on="Product Card Id",
        how="left"
    )
)

In [48]:
plant_product_weekly_demand = (
    plant_product_weekly_demand[
        [
            "order date (DateOrders)",
            "Plant_ID",
            "Product Card Id",
            "Product Name",
            "Order Item Quantity"
        ]
    ]
)

In [49]:
# Order week used for product-level demand monitoring

plant_product_weekly_demand[
    "Order_Date"
] = pd.to_datetime(
    plant_product_weekly_demand[
        "order date (DateOrders)"
    ]
)

In [50]:
# Weekly demand by product within each plant

plant_product_weekly_demand = (
    plant_product_weekly_demand
    .set_index("Order_Date")
    .groupby(
        [
            "Plant_ID",
            "Product Card Id",
            "Product Name"
        ]
    )
    ["Order Item Quantity"]
    .resample("W")
    .sum()
    .reset_index()
)

In [51]:
# Total product demand assigned to a plant during a week

plant_product_weekly_demand = (
    plant_product_weekly_demand.rename(
        columns={
            "Order Item Quantity": "Weekly_Demand"
        }
    )
)

In [52]:
plant_product_weekly_demand.head()

,Plant_ID,Product Card Id,Product Name,Order_Date,Weekly_Demand
0,P1,172,Nike Women's Tempo Shorts,2015-01-04,6
1,P1,172,Nike Women's Tempo Shorts,2015-01-11,11
2,P1,172,Nike Women's Tempo Shorts,2015-01-18,11
3,P1,172,Nike Women's Tempo Shorts,2015-01-25,18
4,P1,172,Nike Women's Tempo Shorts,2015-02-01,4


In [ ]:
'''
plant_product_weekly_demand.to_csv(
    "../data/processed/plant_product_weekly_demand.csv",
    index=False
)
'''